# LittleTales voice bake-off and batch generation (free GPU notebook)

Runs free, openly licensed voice models on a free notebook GPU (for example Google Colab or Kaggle, with a free account, no payment and no API key).

1. **Runtime → Change runtime type → GPU (T4).**
2. Run the setup cell, then upload `bakeoff-lines.json` (or `lines.json`) from `tools/notebooks/` in the project (`npm run audio:export-lines` makes them).
3. Run **one** model section at a time. Clips are saved to `out/` as each one finishes, so a disconnect loses at most one clip; re-running skips finished clips.
4. Download the zip at the end and unpack it into the project's `tools/bakeoff/` (for listening) or `public/` (for real lines).

**Licences (check the current text on the day you use them):** Chatterbox (MIT), Sesame CSM-1B (Apache 2.0), Dia-1.6B (Apache 2.0). Record the model and licence for each clip in `docs/ATTRIBUTIONS.md`.
**Never clone a voice without its owner's written permission, and never clone a child's voice.**
Stay within the service's usage limits and terms, and do not leave the notebook running unattended.

In [ ]:
# Setup
!pip -q install chatterbox-tts soundfile
!pip -q install -U "transformers>=4.54" accelerate
import json, os, zipfile, soundfile as sf, torch
from google.colab import files  # on Kaggle, add the files as a dataset instead
os.makedirs('out', exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (pick a GPU runtime)')
uploaded = files.upload()
LINES = json.load(open(next(iter(uploaded))))
# Bake-off files have {id, text, delivery}; full line lists have {out, text, parts}.
JOBS = [{'name': l.get('id') or l['out'].replace('/', '__'), 'text': l['text'], 'delivery': l.get('delivery') or l['parts'][0]['delivery'], 'out': l.get('out')} for l in LINES]
print(len(JOBS), 'lines')
BATCH = 20  # small batches: run the cell again to continue

## Chatterbox (MIT)
Expressive; `exaggeration` raises energy. A short reference clip can set the voice, only with the speaker's written permission.

In [ ]:
from chatterbox.tts import ChatterboxTTS
cb = ChatterboxTTS.from_pretrained(device='cuda')
STYLE = {'excited': 0.8, 'silly': 0.9, 'surprised': 0.75, 'proud': 0.65, 'whisper': 0.3, 'sleepy': 0.3, 'sad': 0.35, 'gentle': 0.4}
done = 0
for j in JOBS:
    path = f"out/chatterbox__{j['name']}.wav"
    if os.path.exists(path): continue
    wav = cb.generate(j['text'], exaggeration=STYLE.get(j['delivery'], 0.5), cfg_weight=0.4)
    sf.write(path, wav.squeeze().cpu().numpy(), cb.sr)
    done += 1
    if done >= BATCH: break
print('saved', done, 'clips')

## Sesame CSM-1B (Apache 2.0)
Very natural conversational delivery. Needs a free Hugging Face login to accept the model terms (no payment).

In [ ]:
from transformers import CsmForConditionalGeneration, AutoProcessor
proc = AutoProcessor.from_pretrained('sesame/csm-1b')
csm = CsmForConditionalGeneration.from_pretrained('sesame/csm-1b', device_map='cuda')
done = 0
for j in JOBS:
    path = f"out/csm__{j['name']}.wav"
    if os.path.exists(path): continue
    conv = [{'role': '0', 'content': [{'type': 'text', 'text': j['text']}]}]
    inputs = proc.apply_chat_template(conv, tokenize=True, return_dict=True).to('cuda')
    audio = csm.generate(**inputs, output_audio=True)
    proc.save_audio(audio, path)
    done += 1
    if done >= BATCH: break
print('saved', done, 'clips')

## Dia-1.6B (Apache 2.0)
Handles laughs and pauses, e.g. `(laughs)`; slower and heavier.

In [ ]:
from transformers import AutoProcessor, DiaForConditionalGeneration
dproc = AutoProcessor.from_pretrained('nari-labs/Dia-1.6B-0626')
dia = DiaForConditionalGeneration.from_pretrained('nari-labs/Dia-1.6B-0626').to('cuda')
done = 0
for j in JOBS:
    path = f"out/dia__{j['name']}.wav"
    if os.path.exists(path): continue
    inputs = dproc(text=[f"[S1] {j['text']}"], padding=True, return_tensors='pt').to('cuda')
    out = dia.generate(**inputs, max_new_tokens=2048, guidance_scale=3.0, temperature=1.6, top_p=0.9, top_k=45)
    dproc.save_audio(dproc.batch_decode(out), [path])
    done += 1
    if done >= BATCH: break
print('saved', done, 'clips')

## Download

In [ ]:
with zipfile.ZipFile('voices.zip', 'w') as z:
    for f in sorted(os.listdir('out')):
        z.write(os.path.join('out', f), f)
files.download('voices.zip')

## Using the results
- **Bake-off:** unzip into `tools/bakeoff/` and run `npm run audio:bakeoff` to compare models side by side, then note the winner per role in `config/voices.json`.
- **Real lines:** files are named after their target path (`audio__narration__en__...`). `npm run audio:import -- voices.zip --model chatterbox` places them, normalises loudness, writes word timings and marks them so they are never regenerated.